数据操作

首先导入torch。虽然叫pytorch，但是要导入torch

In [1]:
import torch

In [2]:
x = torch.arange(12)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

用.shape来访问张量的形状
用.numel()来访问张量中元素的总数

In [3]:
x.shape

torch.Size([12])

In [4]:
x.numel()

12

如果想只调整张量的形状，而不改变元素数量和元素值，我们可以调用reshape函数

In [5]:
x.reshape(3,4) # reshape成3行4列的矩阵

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [6]:
x.reshape(2,3,2) # reshape成2个3行2列的矩阵

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]]])

创建全零的张量，可以用.zeros(（a,b,c）)
创建全一的张量，可以用.ones(（a,b,c）)

In [7]:
torch.zeros((2,2,3,2)) 

tensor([[[[0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.]]],


        [[[0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [0., 0.]]]])

In [8]:
torch.ones((2,2,3))

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])

也可以直接创建的时候同时确定形状并赋值

In [9]:
torch.tensor([[2,1,3],[4,5,6]]) #list of lists

tensor([[2, 1, 3],
        [4, 5, 6]])

常见的标准算数运算符(+, -, *, /, %, //, **)都可以被升级为按元素运算

In [10]:
x = torch.tensor([1,2])
y = torch.tensor([3,4])
x + y, x - y, x * y, x / y, x%y, x//y, x**y

(tensor([4, 6]),
 tensor([-2, -2]),
 tensor([3, 8]),
 tensor([0.3333, 0.5000]),
 tensor([1, 2]),
 tensor([0, 0]),
 tensor([ 1, 16]))

还有torch.exp(), 括号里面填要操作的tensor

In [11]:
torch.exp(x) # e的x次幂

tensor([2.7183, 7.3891])

还可以把多个张量连接在一起

In [12]:
x = torch.arange(12, dtype=torch.float32).reshape(3,4)
y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((x,y), dim=0), torch.cat((x,y), dim=1)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [13]:
x == y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

In [14]:
x.sum()

tensor(66.)

两个张量即使形状不同，可以通过广播机制，来执行按元素操作

In [15]:
x = torch.arange(3).reshape(3,1)
y = torch.arange(2).reshape(1,2)
x, y

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

In [16]:
z = x + y
z

tensor([[0, 1],
        [1, 2],
        [2, 3]])

如何选择张量中的元素

In [17]:
z[-1], z[0:2], z[:,0:1], z[1,1], z[0:2, 0:2]

(tensor([2, 3]),
 tensor([[0, 1],
         [1, 2]]),
 tensor([[0],
         [1],
         [2]]),
 tensor(2),
 tensor([[0, 1],
         [1, 2]]))

写值的方法（也可以一次对多个位置赋同一个值，对多个位置的索引同上）

In [18]:
z[0,0] = 100
z

tensor([[100,   1],
        [  1,   2],
        [  2,   3]])

运行一些操作可能会导致为新结果分配内存

In [19]:
before = id(x)
x = x + y
id(x) == before

False

In [20]:
before, id(x)

(4662471536, 4662062416)

In [21]:
z = torch.zeros_like(x)
print("id(z):", id(z))
z[:] = x + y
print("id(z):", id(z))

id(z): 4662052336
id(z): 4662052336


如果在后续计算中没有重复使用x, 我们也可以使用x[:] = x + y或x += y来减少操作的内存开销

In [22]:
before = id(x)
x += y
id(x) == before

True

In [23]:
before = id(x)
x[:] = x + y
id(x) == before

True

转换为numpy张量

In [24]:
a = x.numpy()
b = torch.tensor(a)
type(a), type(b)

(numpy.ndarray, torch.Tensor)

将大小为1的张量转化为python标量

In [25]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

数据预处理preprocessing
创建一个人工数据集，并存储在csv（逗号分隔值）文件

In [26]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')

with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')   # 列名
    f.write('NA,Pave,127500\n')         # 每行表示一个样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

从创建的csv文件中加载原始数据集

In [27]:
import pandas as pd
data = pd.read_csv(data_file)
data

ModuleNotFoundError: No module named 'pandas'

为了处理缺失的数据，典型的方法包括插值和删除，这里我们将考虑插值

In [ ]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True)) # 用均值填充缺失值
inputs

,NumRooms,Alley
0,3.0,Pave
1,2.0,NaN
2,4.0,NaN
3,3.0,NaN


对于inputs中的类别值或离散值，我们将“NaN”视为一个类别

In [ ]:
inputs = pd.get_dummies(inputs, dummy_na=True, dtype = int) # 将离散值转成独热编码
inputs

,NumRooms,Alley_Pave,Alley_nan
0,3.0,1,0
1,2.0,0,1
2,4.0,0,1
3,3.0,0,1


把数值类型转化为张量格式

In [ ]:
x, y = torch.tensor(inputs.values, dtype=torch.float32), torch.tensor(outputs.values, dtype=torch.float32)
x, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]]),
 tensor([127500., 106000., 178100., 140000.]))